# **ST 554 Spring 2026 Final Project**
## *Created by Cody Ashby on April 27, 2026*

In [1]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, Binarizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder # set parallelism to 128 when doing CV!
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 20:56:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 20:56:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/27 20:56:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/27 20:56:20 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [2]:
power_data = pd.read_csv("power_ml_data.csv")

In [3]:
spark_power_data = spark.read.csv("power_ml_data.csv",inferSchema=True,header=True)
spark_power_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

In [4]:
sqlTrans=SQLTransformer(statement="SELECT Month, Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, \
                                        Power_Zone_1, Power_Zone_2, Power_Zone_3 as label, CAST(Hour AS DOUBLE) AS Revised_Hour FROM __THIS__")

In [5]:
binary_Hour=Binarizer(threshold=6.5,inputCol="Revised_Hour",outputCol="Binary_Hour")

In [6]:
#Month_Index=StringIndexer(inputCol="Month",outputCol="Indexed_Month")
Month_OHE=OneHotEncoder(inputCol="Month",outputCol="Encoded_Month")

In [7]:
Weather_Vars=VectorAssembler(inputCols=['Temperature','Humidity','Wind_Speed','General_Diffuse_Flows','Diffuse_Flows'],outputCol='weather_features',handleInvalid='keep')
Weather_Comps=Weather_Vars.transform(spark_power_data)
PC_features=PCA(k=2,inputCol='weather_features',outputCol="Prin_Comps")
Fitted_PC=PC_features.fit(Weather_Comps)

In [8]:
total_features=VectorAssembler(inputCols=['Prin_Comps','Binary_Hour','Power_Zone_1','Power_Zone_2','Encoded_Month'],outputCol='features',handleInvalid='keep')

In [9]:
MLR=LinearRegression()
grand_pipeline=Pipeline(stages=[sqlTrans,binary_Hour,Month_OHE,Weather_Vars,PC_features,total_features,MLR])

In [10]:
EN_Param_Grid=ParamGridBuilder().addGrid(MLR.regParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]) \
                            .addGrid(MLR.elasticNetParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]).build()

In [11]:
ElasticNet_CrossVal=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=EN_Param_Grid,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5,
                                   parallelism=128)
ElasticNet_Model=ElasticNet_CrossVal.fit(spark_power_data)

26/04/27 20:57:01 WARN BlockManager: Block rdd_57_0 already exists on this machine; not re-adding it
26/04/27 20:57:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 20:57:04 WARN Instrumentation: [d8aea638] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 20:57:04 WARN Instrumentation: [648ef8b8] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 20:57:04 WARN Instrumentation: [9e8415af] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 20:57:04 WARN Instrumentation: [81ab9215] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 20:57:05 WARN Instrumentation: [5995d227] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 20:57:05 WARN Instrumentation: [15224205] regParam is zero, which might

In [12]:
ElasticNet_RMSEs=[]
for _ in range(len(EN_Param_Grid)):
    ElasticNet_RMSEs.append([ElasticNet_Model.avgMetrics[_],EN_Param_Grid[_].values()])

In [13]:
ElasticNet_RMSEs

[[2147.9335401454514, dict_values([0.0, 0.0])],
 [2147.9335401454514, dict_values([0.0, 0.05])],
 [2147.9335401454514, dict_values([0.0, 0.1])],
 [2147.9335401454514, dict_values([0.0, 0.25])],
 [2147.9335401454514, dict_values([0.0, 0.5])],
 [2147.9335401454514, dict_values([0.0, 0.75])],
 [2147.9335401454514, dict_values([0.0, 0.9])],
 [2147.9335401454514, dict_values([0.0, 0.95])],
 [2147.9335401454514, dict_values([0.0, 0.98])],
 [2147.9335401454514, dict_values([0.0, 0.99])],
 [2147.9335401454514, dict_values([0.0, 1.0])],
 [2147.9335073588495, dict_values([0.05, 0.0])],
 [2147.933818561291, dict_values([0.05, 0.05])],
 [2147.935407331947, dict_values([0.05, 0.1])],
 [2147.931858881458, dict_values([0.05, 0.25])],
 [2147.932520249108, dict_values([0.05, 0.5])],
 [2147.9330645270256, dict_values([0.05, 0.75])],
 [2147.933392787359, dict_values([0.05, 0.9])],
 [2147.933406767649, dict_values([0.05, 0.95])],
 [2147.9334164130264, dict_values([0.05, 0.98])],
 [2147.9334191990833, dict

Upon close inspection of each RMSE associated with each pair of hyperparameters, we can see that the lowest RMSE goes to the one with the `regParam` of 0.25 and the `ElasticNetParam` of 0.05, indicating fairly minor regularization required for this model. The CV error (i.e. the average RMSE) for this model is shown below.

In [14]:
Best_EN_Params=ParamGridBuilder().addGrid(MLR.regParam,[0.25]).addGrid(MLR.elasticNetParam,[0.05]).build()
Best_ElasticNet_CV=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=Best_EN_Params,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5)
Best_ElasticNet_Model=Best_ElasticNet_CV.fit(spark_power_data)
Best_ElasticNet_Model.avgMetrics

[2147.931602118489]

In [25]:
Best_ElasticNet_Model.getEvaluator()

RegressionEvaluator_657886bdc71b

In [27]:
Best_ElasticNet_Model_Preds=Best_ElasticNet_Model.transform(spark_power_data)
RegressionEvaluator.evaluate(Best_ElasticNet_Model_Preds,spark_power_data)

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `_evaluate` is not supported.